# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Javeria-crypto326/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My rule and its reason codes

**Signal check A — staleness.** Bucketed `days_since_last_update` (via `freshness_tier`) against
the decline label: decline rate rises the staler a page gets, in every bucket with a healthy
n → **CONFIRMED**. This is the signal behind FlyRank's real refresh flags.

**Signal check B — search demand.** Bucketed `search_volume` against the decline label: decline
rate barely moves across low/medium/high demand — demand for a keyword doesn't drive whether
*this* page is declining → **MIXED**. A clean negative, not a failed check.

**My rule, in plain words:** *A page is worth a refresh action if it hasn't been updated in a
long time (stale) AND it still gets a meaningful amount of search traffic (visible).*

- `stale` = `days_since_last_update >= 180`
- `visible` = `impressions_90d >= 500`
- **score** = `stale * visible * log(1 + impressions_90d)`
- **reason code** (one only): `"stale_but_visible"` when both conditions fire, else `"no_flag"`
- **action label**: `refresh_now` (rule fired) / `monitor` (visible but not stale) /
  `leave_alone` (neither)

Nothing here touches `trend_direction`, `trend_pct`, or `is_declining_label` — those only appear
below to sanity-check the two signals, never inside the score itself.


In [1]:
# --- Load + clean the data, then check the two signals above ---
import os, subprocess
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/Javeria-crypto326/flyrank-ml-internship.git"
REPO_DIR = "flyrank-ml-internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

for col in ["days_since_last_update", "impressions_90d", "avg_position", "ctr",
            "word_count", "search_volume"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Label used ONLY to test the signals below, never as a rule input.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Signal A: staleness -> decline rate
order_a = ["never", "0-30", "31-90", "91-180", "181+"]
bucket_a = (df.groupby("freshness_tier")["is_declining_label"]
              .agg(decline_rate="mean", n="count")
              .reindex([t for t in order_a if t in df["freshness_tier"].unique()]))
print("Signal A - staleness (freshness_tier) vs decline rate")
print(bucket_a, "\n")

# Signal B: search demand -> decline rate
df["volume_bucket"] = pd.cut(
    df["search_volume"],
    bins=[-0.1, 0, 100, 1000, df["search_volume"].max() + 1],
    labels=["none", "low", "medium", "high"],
)
bucket_b = (df.groupby("volume_bucket", observed=True)["is_declining_label"]
              .agg(decline_rate="mean", n="count"))
print("Signal B - search demand (search_volume) vs decline rate")
print(bucket_b)


Signal A - staleness (freshness_tier) vs decline rate
                decline_rate      n
freshness_tier                     
0-30                0.511377  20480
31-90               0.588571    175
91-180              0.611057   9171
181+                0.471264    174 

Signal B - search demand (search_volume) vs decline rate
               decline_rate      n
volume_bucket                     
none               0.571629  13549
low                0.523877  13402
medium             0.501004   2489
high               0.444643    560


## 2. Build the ranked queue (writes the CSV)

Encode the rule above as a score, attach the one reason code and the action label to every
row, rank the whole table, and write `work/outputs/baseline_action_score.csv`.


In [2]:
# --- 2. Encode the rule, rank everything, write the CSV ---
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["score"] = stale * visible * np.log1p(df["impressions_90d"])
df["reason_code"] = np.where((stale == 1) & (visible == 1), "stale_but_visible", "no_flag")

def action_label(row):
    if row["reason_code"] == "stale_but_visible":
        return "refresh_now"
    if row["impressions_90d"] >= 500:
        return "monitor"
    return "leave_alone"

df["action"] = df.apply(action_label, axis=1)
queue = df.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id", "client_id", "score", "reason_code", "action",
            "days_since_last_update", "impressions_90d", "avg_position", "ctr"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Wrote work/outputs/baseline_action_score.csv")
print("Rows:", len(queue), "| flagged refresh_now:", (queue["action"] == "refresh_now").sum())
queue[out_cols].head(10)


Wrote work/outputs/baseline_action_score.csv
Rows: 30000 | flagged refresh_now: 17


,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr
0,content_cf56e2e2e282,client_7f2253d7e2,11.029699,stale_but_visible,refresh_now,194,61678,19.7,0.15
1,content_7368877ea310,client_7f2253d7e2,10.993278,stale_but_visible,refresh_now,194,59472,24.8,0.13
2,content_1bfaa38ff26c,client_7f2253d7e2,10.154869,stale_but_visible,refresh_now,194,25715,22.2,0.23
3,content_0a91db491d14,client_7f2253d7e2,9.495519,stale_but_visible,refresh_now,193,13299,10.5,0.49
4,content_5feee3994adb,client_7f2253d7e2,8.963544,stale_but_visible,refresh_now,194,7812,39.0,0.01
5,content_c2d929d83eaa,client_7f2253d7e2,8.930494,stale_but_visible,refresh_now,193,7558,17.9,0.20
6,content_b16bd7307b39,client_7f2253d7e2,8.431853,stale_but_visible,refresh_now,194,4590,31.0,0.00
7,content_fe16a55cd13d,client_7f2253d7e2,8.424420,stale_but_visible,refresh_now,194,4556,16.4,0.33
8,content_ecb6215e79fd,client_7f2253d7e2,8.396155,stale_but_visible,refresh_now,194,4429,25.3,0.38
9,content_928af3e22c80,client_7f2253d7e2,7.437206,stale_but_visible,refresh_now,193,1697,15.8,0.12


## 3. Top-10 review

*(Ten per this week's card — top-20 is the optional deeper version.)* For each of the top ten:
the action, why it's there, and what would make it wrong — built straight from that row's own
numbers so it stays honest rather than guessed after the fact.


In [3]:
# --- 3. Top-10 hand review, generated from the data itself ---
top10 = queue.head(10)

for i, row in top10.iterrows():
    why = (f"stale ({row['days_since_last_update']:.0f} days since update) "
           f"and still visible ({row['impressions_90d']:.0f} impressions/90d)")
    wrong_bits = []
    if row["avg_position"] > 0 and row["avg_position"] <= 3:
        wrong_bits.append("it's already ranking top-3, so a refresh could be unnecessary risk")
    if 0 < row["ctr"] and row["ctr"] >= 2:
        wrong_bits.append("CTR already looks healthy, so the traffic problem may not be the content")
    if row["avg_position"] == 0:
        wrong_bits.append("avg_position is 0 (no position data) - the visibility read may be shaky")
    if not wrong_bits:
        wrong_bits.append("the staleness could be a false alarm if the page was quietly updated "
                           "after this export was taken")
    wrong = "; ".join(wrong_bits)

    print(f"{i+1}. content_id={row['content_id']} | action={row['action']}")
    print(f"   why: {why}")
    print(f"   what would make it wrong: {wrong}\n")


1. content_id=content_cf56e2e2e282 | action=refresh_now
   why: stale (194 days since update) and still visible (61678 impressions/90d)
   what would make it wrong: the staleness could be a false alarm if the page was quietly updated after this export was taken

2. content_id=content_7368877ea310 | action=refresh_now
   why: stale (194 days since update) and still visible (59472 impressions/90d)
   what would make it wrong: the staleness could be a false alarm if the page was quietly updated after this export was taken

3. content_id=content_1bfaa38ff26c | action=refresh_now
   why: stale (194 days since update) and still visible (25715 impressions/90d)
   what would make it wrong: the staleness could be a false alarm if the page was quietly updated after this export was taken

4. content_id=content_0a91db491d14 | action=refresh_now
   why: stale (193 days since update) and still visible (13299 impressions/90d)
   what would make it wrong: the staleness could be a false alarm if the pa

## 4. Weak picks + leakage check

Which top picks look wrong, and proof no product flags or future/label-derived columns leaked
into the rule.


In [4]:
# --- 4. Leakage check + weak picks ---
banned_columns = {"trend_direction", "trend_pct", "is_declining_label"}
score_inputs = {"days_since_last_update", "impressions_90d"}
print("Score built only from:", score_inputs)
print("Any overlap with banned/label columns?", bool(score_inputs & banned_columns))

weak = top10[(top10["avg_position"] > 0) & (top10["avg_position"] <= 3)]
print(f"\nWeak picks in the top 10 (already top-3, so 'refresh_now' may be overkill): {len(weak)}")
if len(weak):
    print(weak[["content_id", "avg_position", "impressions_90d", "action"]])


Score built only from: {'impressions_90d', 'days_since_last_update'}
Any overlap with banned/label columns? False

Weak picks in the top 10 (already top-3, so 'refresh_now' may be overkill): 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
